<a href="https://colab.research.google.com/github/wavymejti/KursAI1/blob/main/naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import json
import random
import math

# wczytyanie danych treningowych

def load_email_data(spam_path, ham_path):
  with open(spam_path, encoding='utf-8') as f1, open(ham_path, encoding='utf-8') as f2:
    spam_data = json.load(f1)
    ham_data = json.load(f2)
  return spam_data + ham_data



def train_test_split(data, test_ratio=0.2):
  random.shuffle(data)
  cut = int(len(data) * (1 - test_ratio))
  return data[:cut], data[cut:]


In [39]:
import pickle
import base64
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

def get_gmail_service():
  with open("token.pkl", "rb") as token:
    creds = pickle.load(token)
  if creds and creds.expired and creds.refresh_token:
    creds.refresh(Request())
  service = build("gmail", "v1", credentials=creds)
  return service

def get_label_id(service, label_name):
  labels = service.users().labels().list(userId="me").execute().get('labels', [])
  print(f"Dostępne etykiety (nazwy): {[label['name'] for label in labels]}")
  for label in labels:
    if label["name"].lower() == label_name.lower(): # Zmieniono na porównanie niewrażliwe na wielkość liter
      print(f"Znaleziono etykietę: {label['name']} z ID: {label.get('id')}") # Debug print
      return label["id"]
  print(f"Etykieta '{label_name}' nie została znaleziona w funkcji get_label_id.") # Debug print if not found
  return None

def fetch_unread_emails_from_label(label_name='Test'):
  service = get_gmail_service()
  label_id = get_label_id(service, label_name)
  if not label_id:
    print(f"Etykieta '{label_name}' nie została znaleziona.")
    return []
  response = service.users().messages().list(userId='me',labelIds=[label_id, 'UNREAD'],maxResults=100).execute()
  messages = response.get('messages', [])
  email_list = []

  for msg in messages:
    msg_id = msg['id']
    message = service.users().messages().get(userId='me', id=msg_id, format = 'full').execute()
    payload = message.get('payload', {}) # Zmieniono z ['payload'] na 'payload'
    headers = payload.get('headers', []) # Zmieniono z ['headers'] na 'headers'

    subject = next((header['value'] for header in headers if header['name'] == 'Subject'), None)
    body = get_message_body(payload)
    email_list.append({'subject': subject, 'body': body})

  return email_list
def get_message_body(payload):
  parts = payload.get('parts')
  if parts:
    for part in parts:
      if part['mimeType'] == 'text/plain':
        data = part['body'].get('data')
        if data:
          return base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
        else:
          # 'body_data' is not defined here, likely a typo. Assuming it should be data from current part if data was None.
          # Given the outer 'if data:', this else block might not be reached if data is consistently None,
          # or it might be intended for a different structure. For now, returning empty string if no data.
          return ""
  return "brak tresci"


In [40]:
# trenowanie klasyfikatora bayesa
def preprocess(text):
  return text.lower().replace("-", "").replace(".", " ").replace(",", " ")\
  .replace("!", " ").replace("?", " ").split()

def train_naive_bayes(train_data, alpha=1.0):
  class_counts = {}
  word_counts = {}
  total_words = {}

  for rec in train_data:
    label = rec["etykieta"]
    class_counts[label] = class_counts.get(label , 0) + 1
    word_counts.setdefault(label, {})
    total_words.setdefault(label, 0)

    words = preprocess(rec["tekst"])
    for word in words:
      word_counts[label][word] = word_counts[label].get(word, 0) + 1
      total_words[label] += 1

  vocab = set()
  for wc in word_counts.values():
    vocab.update(wc.keys())

  return {
      "class_counts": class_counts,
      "word_counts": word_counts,
      "total_words": total_words,
      "vocab": vocab,
      "alpha": alpha,
      "total_docs": len(train_data)
  }

#klasyfikacja wiadomosci
def log_prob(model, words, class_name):
  logp = math.log(model["class_counts"][class_name] / model["total_docs"])
  V = len(model["vocab"])
  a = model["alpha"]
  for word in words:
    wc = model["word_counts"][class_name].get(word, 0)
    logp += math.log((wc + a) / (model["total_words"][class_name] + a * V))
  return logp

def predict(model, text):
  words = preprocess(text)
  best_class, best_log = None, -float("inf")
  for class_name in model["class_counts"]:
    logp = log_prob(model, words, class_name)
    if logp > best_log:
      best_log = logp
      best_class = class_name

  return best_class

def evaluate(model, test_data):
  correct = 0
  for rec in test_data:
    prediction = predict(model, rec["tekst"])
    if prediction == rec["etykieta"]:
      correct += 1
  accuracy = correct / len(test_data)
  print(f"skutecznosc na zbiorze testowym: {accuracy * 100:.2f}%")
  return accuracy



In [41]:
# główna funkcja
from pprint import pprint

def main():
  data = load_email_data("spam.json", "ham.json")

  train, test = train_test_split(data)

  model = train_naive_bayes(train)

  #pprint(model)

  evaluate(model, test)

  while True:
    text = input("Podaj tekst wiadomosci: ")
    prediction = predict(model, text)
    print(f"Przewidywana etykieta: {prediction}")

# uruchomienie
if __name__ == "__main__":
  emails = fetch_unread_emails_from_label('TEST')
  for i, email in enumerate(emails, 1):
    print(f"\n---Wiadomość {i} ---\n{email}")


Dostępne etykiety (nazwy): ['CHAT', 'SENT', 'INBOX', 'IMPORTANT', 'TRASH', 'DRAFT', 'SPAM', 'CATEGORY_FORUMS', 'CATEGORY_UPDATES', 'CATEGORY_PERSONAL', 'CATEGORY_PROMOTIONS', 'CATEGORY_SOCIAL', 'YELLOW_STAR', 'STARRED', 'UNREAD', 'TEST', 'SPAM2', 'HAM']
Znaleziono etykietę: TEST z ID: Label_466888836789425889

---Wiadomość 1 ---
{'subject': '[KOD ZNIŻKOWY] Nowości wydawnicze w tym miesiącu 📚', 'body': 'Cześć! Przygotowaliśmy dla Ciebie zestawienie najciekawszych premier\r\nliterackich kwietnia. Specjalnie dla naszych subskrybentów mamy kod:\r\n*CZYTAM20*, który obniża cenę wybranych tytułów o 20%. Miłej lektury!\r\n'}

---Wiadomość 2 ---
{'subject': 'Twoje zamówienie nr 84920 zostało przekazane do realizacji', 'body': 'Witaj Janie, Dziękujemy za zakupy w naszym sklepie! Twoja płatność została\r\nzaksięgowana. Jak tylko przekażemy paczkę kurierowi, otrzymasz od nas\r\nkolejną wiadomość z numerem listu przewozowego do śledzenia przesyłki.\r\nZespół Sklepu X.\r\n'}

---Wiadomość 3 ---
{'s